# Workstream 4: Segmentation evaluation and ablations

Patient-level 3D evaluation of the frozen-DINOv3 BraTS baseline and of every later variant
(uniform / adaptive token merging from Workstream 2, reduced resolution from Workstream 3).

What is measured (see `docs/EVALUATION.md`):

* **Overlap**: Dice, sensitivity, precision, per patient and per region (WT, TC, ET).
* **Boundary**: HD95, ASSD, normalized surface Dice at 1 mm and 2 mm.
* **Where errors are**: distance bands from the tumor boundary, and 16x16 token patches that are
  background / straddle the boundary / fully tumor.
* **Small lesions**: detection rate of ground-truth components by size; false-positive components.
* **Statistics**: paired per-patient differences vs a reference, bootstrap 95% CI, Wilcoxon + Holm.

Rules: validation split only. The 188-patient test split stays untouched until the final, fixed protocol.

## 1. Setup

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_ROOT = Path("/content/CAP6411-Adaptive-Token-Merging")
BRANCH = "feature/evaluation"  # switch to "main" after the PR is merged
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH,
                    "https://github.com/FaramarzAboutalebi/CAP6411-Adaptive-Token-Merging.git",
                    str(REPO_ROOT)], check=True)
%cd {REPO_ROOT}
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q --ff-only origin {BRANCH}
!git log --oneline -1
%pip install -q -r requirements.txt
%pip install -q -e . --no-deps
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

### Weight-free check (no data, no GPU)

In [ ]:
!python -m unittest tests.test_evaluation_metrics tests.test_evaluation_pipeline

## 2. Private paths and the trained baseline

Needs four private files. Nothing here is written to Git.

* `workstream1_epoch2/` (shared by Faramarz): the trained adapter/decoder artifact. Shared items are not visible
  to Colab until you right-click them in Drive -> **Organize -> Add shortcut** -> My Drive.
* `brats2023_gli_split_seed42.json` (shared by Faramarz): the patient split. Add a shortcut the same way.
* The BraTS 2023 GLI training ZIP and the gated DINOv3 ViT-B/16 checkpoint: your own authorized copies
  (not in the shared package). Edit the two paths below to wherever they are.

In [ ]:
import hashlib
from google.colab import drive
drive.mount("/content/drive")

MY_DRIVE = Path("/content/drive/MyDrive")
ARTIFACT = MY_DRIVE / "workstream1_epoch2/dinov3_vitb16_brats_epoch2_adapter_decoder.pt"
MANIFEST = MY_DRIVE / "brats2023_gli_split_seed42.json"
ARCHIVE = MY_DRIVE / "CAP6411/datasets/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData.zip"  # edit
DINO_CKPT = MY_DRIVE / "CAP6411/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"  # edit
RESULTS = MY_DRIVE / "CAP6411/results/workstream4"  # your own space; contains patient IDs, do not share publicly
FIGURES = RESULTS / "figures"
INSPECTION = REPO_ROOT / "configs/brats2023_gli_inspection.json"
MODEL_CONFIG = REPO_ROOT / "configs/dinov3_vitb16_brats.json"
for path in (ARCHIVE, MANIFEST, DINO_CKPT, ARTIFACT):
    print(f"{'OK     ' if path.is_file() else 'MISSING'} {path}")

ARTIFACT_SHA256 = "ed0d84caf6bb6d4dcb4ee2dc29dc911c628406204e9683e7cde3b280b796314b"  # docs/WORKSTREAM1_ARTIFACTS.md
assert hashlib.sha256(ARTIFACT.read_bytes()).hexdigest() == ARTIFACT_SHA256, "artifact checksum mismatch"
print("artifact checksum OK")

In [ ]:
DINO_ROOT = Path("/content/dinov3")
DINO_COMMIT = "6876159a11b4df116f30f667f8c9888617df0751"
if not DINO_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/facebookresearch/dinov3.git", str(DINO_ROOT)], check=True)
subprocess.run(["git", "-C", str(DINO_ROOT), "checkout", "-q", DINO_COMMIT], check=True)
%pip install -q -e /content/dinov3 --no-deps
%pip install -q ftfy omegaconf regex scikit-learn submitit termcolor torchmetrics

In [ ]:
import torch
from cap6411_atm.models import DINOv3SegmentationConfig, build_pretrained_brats_dinov3, load_adapter_decoder_artifact

device = torch.device("cuda")
print(torch.cuda.get_device_name(0))
model_config = DINOv3SegmentationConfig.from_json(MODEL_CONFIG)
model = build_pretrained_brats_dinov3(DINO_ROOT, DINO_CKPT, model_config)
artifact_meta = load_adapter_decoder_artifact(model, ARTIFACT)
model = model.to(device).eval()
print("Artifact validation Macro Dice:", artifact_meta["validation"]["macro_dice"])

## 3. Baseline evaluation: one forward pass, many decision rules

`QUICK = True` evaluates the first 20 validation patients (~2 min) for iterating on analysis code.
Set `QUICK = False` for reported numbers (all 187 patients, ~11 min forward + ~1 s/patient/rule metrics on a T4).
Results are resumable: a finished variant is skipped on re-run.

In [ ]:
import json
from cap6411_atm.evaluation import (
    POSTPROCESSING_ABLATION, VariantSpec, build_eval_loader, build_provenance, collect_results, run_ablation,
    threshold_sweep,
)

QUICK = True
manifest = json.loads(MANIFEST.read_text())
validation_ids = sorted(manifest["splits"]["validation"])
patients = validation_ids[:20] if QUICK else None
run_dir = RESULTS / ("quick20" if QUICK else "validation187")

loader = build_eval_loader(ARCHIVE, MANIFEST, INSPECTION, split="validation", patients=patients, batch_size=16)

rules = {**POSTPROCESSING_ABLATION, **threshold_sweep([0.3, 0.4, 0.6, 0.7])}
variants = [VariantSpec("baseline", method="baseline", merge_ratio=0.0, rules=rules)]
models = {"baseline": model}

# Fingerprints of every input; finished results are reused only if these still match.
provenance = build_provenance(
    checkpoint={"dinov3": DINO_CKPT, "adapter_decoder": ARTIFACT},
    dataset=ARCHIVE, manifest=MANIFEST, inspection_config=INSPECTION, model_config=MODEL_CONFIG.name,
)

frame = run_ablation(variants, lambda spec: models[spec.name], loader, run_dir, provenance=provenance, device=device)
print(frame["method"].value_counts())

In [ ]:
from cap6411_atm.evaluation import global_dice

table = global_dice(frame)
display(table.round(4))
if not QUICK:
    reproduced = table.loc["baseline", "macro"]
    print(f"Reproduced global Macro Dice {reproduced:.6f} vs Workstream 1 0.819176")
    assert abs(reproduced - 0.819176) < 2e-3, "evaluation does not reproduce the Workstream 1 number"

## 4. Patient-level tables (report Table: baseline and decision rules)

In [ ]:
from cap6411_atm.evaluation import summarize, paired_comparison, format_table

summary = summarize(frame, metrics=["dice", "hd95", "nsd_2mm", "sensitivity", "precision"])
print(format_table(summary, metrics=["dice", "hd95", "nsd_2mm"]))
summary.to_csv(run_dir / "summary_table.csv", index=False)
(run_dir / "summary_table.tex").write_text(format_table(summary, metrics=["dice", "hd95", "nsd_2mm"], latex=True))

In [ ]:
comparison = paired_comparison(frame, "baseline", metrics=["dice", "hd95", "nsd_2mm"])
cols = ["metric", "region", "method", "mean_diff", "ci_low", "ci_high", "better", "worse", "p_holm"]
display(comparison[cols].round(4))
comparison.to_csv(run_dir / "paired_vs_baseline.csv", index=False)

## 5. Where do the errors live? (boundary bands, token patches, small lesions)

In [ ]:
from cap6411_atm.evaluation.statistics import boundary_profile, patch_profile, lesion_detection
from cap6411_atm.evaluation import visualization as viz

methods = ["baseline", "baseline/nested", "baseline/nested_cc50_et_total200"]
sub = frame[frame["method"].isin(methods)]
for region in ("WT", "ET"):
    bands = boundary_profile(sub, region=region)
    display(bands.round(3))
    viz.save_figure(viz.plot_profile(bands, ylabel="share of errors", title=f"{region}: errors by distance to GT boundary"),
                    FIGURES / f"bands_{region}")
patches = patch_profile(sub, region="WT")
display(patches)
viz.save_figure(viz.plot_profile(patches, ylabel="error rate", title="WT: error rate per 16x16 token patch type"),
                FIGURES / "patch_profile_WT")
display(lesion_detection(sub, region="ET").round(3))

## 6. Tumor-size strata and failure cases

In [ ]:
from cap6411_atm.evaluation.statistics import stratify_by_tumor_size, failure_cases

display(stratify_by_tumor_size(sub, size_region="WT", metric="dice", metric_region="ET").round(3))
worst = sub[(sub["method"] == "baseline") & (sub["region"] == "ET")].nsmallest(8, "dice")[
    ["patient_id", "dice", "hd95", "gt_volume", "pred_volume", "status"]]
display(worst)

In [ ]:
from cap6411_atm.evaluation import evaluate_model

figure_patients = list(worst["patient_id"].head(3))
fig_loader = build_eval_loader(ARCHIVE, MANIFEST, INSPECTION, split="validation", patients=figure_patients, batch_size=16)
shown = evaluate_model(model, fig_loader, rules={k: POSTPROCESSING_ABLATION[k] for k in ("raw", "nested_cc50_et_total200")},
                       device=device, save_patients=figure_patients)
for pid, saved in shown.saved_predictions.items():
    z = viz.pick_slice(saved["target"], "WT")
    fig = viz.plot_case(saved["image"][..., z], saved["target"][..., z],
                        {name: saved[name][..., z] for name in ("raw", "nested_cc50_et_total200")},
                        region="ET", title=f"slice {int(saved['slices'][z])}")
    viz.save_figure(fig, FIGURES / f"case_{pid}")  # figures stay in restricted Drive storage

## 7. Plugging in Workstream 2 / 3 variants

Each variant is a `VariantSpec` plus a `build_model(spec)` that returns a callable producing `[B,3,240,240]` logits.
Keep data, rules and `MetricConfig` identical across variants. For matched-latency plots pass Workstream 3's measured
latencies as `x_values={method: ms}`.

In [ ]:
from cap6411_atm.evaluation import merge_ratio_grid

RATIOS = [0.1, 0.25, 0.4, 0.5]
merge_variants = merge_ratio_grid("uniform", RATIOS) + merge_ratio_grid("adaptive", RATIOS)

def build_model(spec):
    if spec.method == "baseline":
        return model
    # Example once Workstream 2 lands (names are placeholders until their API exists):
    # from cap6411_atm.merging import build_merged_segmenter
    # return build_merged_segmenter(model, method=spec.method, ratio=spec.merge_ratio, **spec.params).to(device).eval()
    raise NotImplementedError(f"{spec.method} merging is not available yet")

RUN_MERGING = False
if RUN_MERGING:
    frame = run_ablation(variants + merge_variants, build_model, loader, run_dir, provenance=provenance, device=device)
    raw = frame[frame["rule"] == "raw"]
    for region in ("WT", "TC", "ET"):
        for metric in ("dice", "hd95", "nsd_2mm"):
            viz.save_figure(viz.plot_metric_vs_x(raw, metric=metric, region=region, xlabel="merge ratio"),
                            FIGURES / f"{metric}_{region}_vs_ratio")